# Lane 1: Symplectic Muon - Google Colab Runner
**Project:** Symplectic Muon (Hamiltonian Matrix-Orthogonalized Optimization)  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Does thermostatted symplectic kinetic scaling converge stably on an ill-conditioned spectrum and eliminate late-stage limit-cycle oscillations?

This notebook is configured for one-click execution on Google Colab (Free T4 or Pro A100/L4) or local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification
import sys, os, time, math
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB | bfloat16: {bf16_ok}")
    # Optimize execution for NVIDIA Tensor Cores
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Environment Setup & Fast Dependency Installation
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Configuring Google Colab environment...")
    !pip install -q uv
    !uv pip install --system -q pyyaml wandb pytest
    
    # If working directly in Colab without local git clone:
    if not os.path.exists("src"):
        # Create minimal package structure if running standalone
        os.makedirs("src/symplectic_muon", exist_ok=True)
        os.makedirs("configs", exist_ok=True)
        os.makedirs("results", exist_ok=True)

if "src" not in sys.path:
    sys.path.insert(0, "src")
print("Environment ready.")


In [ ]:
# Step 3: Fast Symplectic Muon Engine
from symplectic_muon.optimizer import SymplecticMuon, zeropower_via_newtonschulz5
from symplectic_muon.spectral import compute_spectral_metrics

print("Symplectic Muon engine loaded successfully.")


In [ ]:
# Step 4: Execute Smoke Test on Ill-Conditioned Spectrum
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(2026)
d_in, d_hidden, d_out = 128, 256, 64
batch_size, n_steps = 64, 500

# Synthesize ill-conditioned spectrum with geometric singular decay
X = torch.randn(batch_size * 10, d_in, device=device)
decay = torch.pow(0.8, torch.arange(d_in, device=device).float())
X = X * decay
W_true = torch.randn(d_in, d_out, device=device)
Y = X @ W_true + 0.05 * torch.randn(X.size(0), d_out, device=device)

model = nn.Sequential(
    nn.Linear(d_in, d_hidden, bias=False),
    nn.GELU(),
    nn.Linear(d_hidden, d_out, bias=False)
).to(device)

opt = SymplecticMuon(
    model.parameters(),
    lr=0.01,
    momentum=0.95,
    gamma_min=0.5,
    gamma_max=1.8,
    ema_alpha=0.05,
    steps=5
)

t0 = time.time()
losses = []
for step in range(n_steps):
    idx = (step * batch_size) % (X.size(0) - batch_size)
    x_b, y_b = X[idx:idx+batch_size], Y[idx:idx+batch_size]
    
    pred = model(x_b)
    loss = F.mse_loss(pred, y_b)
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    losses.append(loss.item())
    if step % 100 == 0 or step == n_steps - 1:
        print(f"Step {step:03d} | Loss: {loss.item():.4f}")

elapsed = time.time() - t0
print(f"\nOptimization finished in {elapsed:.2f}s ({elapsed/n_steps*1000:.2f} ms/step).")


In [ ]:
# Step 5: Pre-Registered Gate Assessment
initial_loss = losses[0]
final_loss = losses[-1]
passed = final_loss < initial_loss * 0.10

print("=" * 60)
print(f"Initial Loss: {initial_loss:.4f} -> Final Loss: {final_loss:.4f}")
print(f"Loss Reduction: {(1.0 - final_loss/initial_loss)*100:.1f}%")
print(f"Pre-Registered Gate (final_loss < initial_loss * 0.10): {'PASSED (Keep)' if passed else 'FAILED (Kill)'}")
print("=" * 60)
